# **Toma de decisiones organizacionales**
### **Realizado por** Juan Esteban Gamba, Sofia Forero Estupiñan y Jose Gabriel Vega Forero

El dataset Emergency - 911 Calls contiene 663.523 registros de llamadas de emergencias en el condado de Montgomery, Pennsylvania, entre el 2015 y el 2020.

Los datos se obtuvieron de Kaggle en el enlace: https://www.kaggle.com/datasets/mchirico/montcoalert/data



In [13]:
# Descomentar solo si se ejecuta en Google Colab (no hay venv local):
!pip install -q pandas gdown openpyxl matplotlib seaborn plotly osmnx networkx folium scikit-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 4.3 MB/s eta 0:00:00


In [14]:
import os
import pandas as pd

try:
    import gdown
    output_name = '911.csv'
    if os.path.exists(output_name):
        print(f"{output_name} ya existe. Se omite la descarga.")
    else:
        file_id = '1IM0RMXPC9yQvp7jLXY3Toqix0A7oHzPT'
        gdown.download(id=file_id, output=output_name)
        print("Archivo descargado desde Google Drive.")
    df = pd.read_csv(output_name, header=None)
except Exception as e:
    print(f"No se pudo descargar/leer desde Drive ({e}). Leyendo archivo local...")
    df = pd.read_csv('911_calls_dataset.csv', header=None)

print("📊 Primeros 10 registros del archivo (con columnas asignadas):")
print(df.head(10))

#output_excel = 'BASE.xlsx'
#df.to_excel(output_excel, index=False)
#files.download(output_excel)

911.csv ya existe. Se omite la descarga.


/tmp/ipykernel_2520/2377561500.py:13: DtypeWarning:

Columns (0,1,3,8) have mixed types. Specify dtype option on import or set low_memory=False.



📊 Primeros 10 registros del archivo (con columnas asignadas):
            0            1                                                  2  \
0         lat          lng                                               desc   
1  40.2978759  -75.5812935  REINDEER CT & DEAD END;  NEW HANOVER; Station ...   
2  40.2580614  -75.2646799  BRIAR PATH & WHITEMARSH LN;  HATFIELD TOWNSHIP...   
3  40.1211818  -75.3519752  HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...   
4  40.1161530  -75.3435130  AIRY ST & SWEDE ST;  NORRISTOWN; Station 308A;...   
5  40.2514920  -75.6033497  CHERRYWOOD CT & DEAD END;  LOWER POTTSGROVE; S...   
6  40.2534732  -75.2832450  CANNON AVE & W 9TH ST;  LANSDALE; Station 345;...   
7  40.1821111  -75.1277951  LAUREL AVE & OAKDALE AVE;  HORSHAM; Station 35...   
8  40.2172859  -75.4051820  COLLEGEVILLE RD & LYWISKI RD;  SKIPPACK; Stati...   
9  40.2890267  -75.3995896  MAIN ST & OLD SUMNEYTOWN PIKE;  LOWER SALFORD;...   

       3                           4          

#Dimension de dataset

1. Preparación de datos

Se realizo una conversión de variables y limpieza de las data

In [17]:
column_names = df.iloc[0].values
df.columns = column_names
# Convertir a float
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lng'] = pd.to_numeric(df['lng'], errors='coerce')
# Convertir a datetime
df['timeStamp'] = pd.to_datetime(df['timeStamp'], errors='coerce')
# Eliminar
df = df.drop(columns=['e'])
df.info()

/tmp/ipykernel_2520/1582190210.py:7: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 663523 entries, 0 to 663522
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   lat        663522 non-null  float64       
 1   lng        663522 non-null  float64       
 2   desc       663523 non-null  object        
 3   zip        583324 non-null  object        
 4   title      663523 non-null  object        
 5   timeStamp  663522 non-null  datetime64[ns]
 6   twp        663230 non-null  object        
 7   addr       663523 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 40.5+ MB


AHP

In [36]:
df_ems = df[df['title'].str.contains('EMS', na=False)]
print("Primeros 5 registros del nuevo DataFrame filtrado por 'EMS' en el título:")
print(df_ems.head())

Primeros 5 registros del nuevo DataFrame filtrado por 'EMS' en el título:
Columnas        lat        lng  \
1         40.297876 -75.581294   
2         40.258061 -75.264680   
4         40.116153 -75.343513   
5         40.251492 -75.603350   
6         40.253473 -75.283245   

Columnas                                               desc    zip  \
1         REINDEER CT & DEAD END;  NEW HANOVER; Station ...  19525   
2         BRIAR PATH & WHITEMARSH LN;  HATFIELD TOWNSHIP...  19446   
4         AIRY ST & SWEDE ST;  NORRISTOWN; Station 308A;...  19401   
5         CHERRYWOOD CT & DEAD END;  LOWER POTTSGROVE; S...    NaN   
6         CANNON AVE & W 9TH ST;  LANSDALE; Station 345;...  19446   

Columnas                    title           timeStamp                twp  \
1          EMS: BACK PAINS/INJURY 2015-12-10 17:10:52        NEW HANOVER   
2         EMS: DIABETIC EMERGENCY 2015-12-10 17:29:21  HATFIELD TOWNSHIP   
4          EMS: CARDIAC EMERGENCY 2015-12-10 16:47:36         NORRISTOWN

In [39]:
df_ems_titles = df_ems[['title']].drop_duplicates()
print("Primeros 5 títulos únicos de EMS:")
print(df_ems_titles.head())

Primeros 5 títulos únicos de EMS:
Columnas                    title
1          EMS: BACK PAINS/INJURY
2         EMS: DIABETIC EMERGENCY
4          EMS: CARDIAC EMERGENCY
5                  EMS: DIZZINESS
6                EMS: HEAD INJURY


In [48]:
import pandas as pd

# Crear el DataFrame base de servicios únicos (excluyendo el encabezado incorrecto 'title')
unique_services = [s for s in df['Service'].unique() if s != 'title']
df_services_with_params = pd.DataFrame(unique_services, columns=['Unique_Service'])

# Mapeo clínico sugerido de acuerdo a los criterios ESI proporcionados
esi_mapping = {
    'CARDIAC EMERGENCY': 2,
    'RESPIRATORY EMERGENCY': 2,
    'HEMOPHILIA': 2,
    'AMPUTATION': 2,
    'CHOKING': 2,
    'ASPHYXIATION': 2,
    'DROWNING': 2,
    'ELECTROCUTION': 2,
    'OVERDOSE': 2,
    'VEHICLE ACCIDENT': 2,
    'GUNSHOT': 2,
    'STABBING': 2,
    'TRAUMA INJURY': 2,
    'SUDDEN DEATH': 2,
    'SEIZURES': 2,
    'STROKE': 2,
    'ALLERGIC REACTION': 2,
    'UNCONSCIOUS SUBJECT': 2,
    'ACTIVE SHOOTER': 2,
    'BARRICADED SUBJECT': 2,
    'BOMB DEVICE': 2,

    'DIABETIC EMERGENCY': 3,
    'SYNCOPAL EPISODE': 3,
    'SUBJECT IN PAIN': 3,
    'ABDOMINAL PAINS': 3,
    'BACK PAINS/INJURY': 3,
    'HEAD INJURY': 3,
    'FRACTURE': 3,
    'BURNS': 3,
    'DEHYDRATION': 3,
    'FEVER': 3,
    'MATERNITY': 3,
    'OBSTETRICS': 3,
    'CHEST PAINS': 2, # ESI 2 por sospecha de infarto

    'DIZZINESS': 4,
    'FALL VICTIM': 4,
    'GENERAL WEAKNESS': 4,
    'NAUSEA/VOMITING': 4,
    'EYE INJURY': 4,
    'COLD': 4,
    'ANIMAL BITE': 4,
    'Lacerations': 4,
    'SICK PERSON': 4,
    'HEAT EXHAUSTION': 4,

    'RESCUE - GENERAL': 3,
    'FIRE ALARM': 3,
    'GAS-ODOR/LEAK': 3,
    'CARBON MONOXIDE DETECTOR': 3,
    'ELECTRICAL FIRE OUTSIDE': 3,
    'BUILDING FIRE': 3,
    'VEHICLE FIRE': 3,
    'WOODS/FIELD FIRE': 3,

    'PUBLIC SERVICE': 5,
    'FOOT PATROL': 5,
    'POLICE INFORMATION': 5,
    'DISABLED VEHICLE': 5,
    'ROAD OBSTRUCTION': 5,
    'HAZARDOUS ROAD CONDITIONS': 5,
    'ELEVATOR EMERGENCY': 4,
    'STANDBY': 5,
    'UNKNOWN MEDICAL EMERGENCY': 3
}

# Asignar la clasificación ESI utilizando el mapeo (por defecto 3 si no se especifica explícitamente)
df_services_with_params['ESI'] = df_services_with_params['Unique_Service'].map(esi_mapping).fillna(3).astype(int)

# Inicializar el resto de parámetros en cero
df_services_with_params['Dolor'] = 0
df_services_with_params['parametro3'] = 0
df_services_with_params['parametro4'] = 0

# Mostrar el resultado final ordenado por prioridad ESI sin la fila 'title'
display(df_services_with_params.sort_values(by='ESI').head(96))

,Unique_Service,ESI,Dolor,parametro3,parametro4
3,CARDIAC EMERGENCY,2,0,0,0
7,RESPIRATORY EMERGENCY,2,0,0,0
10,VEHICLE ACCIDENT,2,0,0,0
30,OVERDOSE,2,0,0,0
25,SEIZURES,2,0,0,0
...,...,...,...,...,...
76,POLICE INFORMATION,5,0,0,0
88,PUBLIC SERVICE,5,0,0,0
91,HAZARDOUS ROAD CONDITIONS,5,0,0,0
92,FOOT PATROL,5,0,0,0


In [20]:
import os
import pandas as pd
import gdown

output_name = 'servicios_con_parametros (2).csv'

file_id = '1DZBVa1gFulSxMx3fkyUxScjWg9vvc6sz'

try:
    if os.path.exists(output_name):
        print(f"{output_name} ya existe. Se omite la descarga.")
    else:
        gdown.download(
            id=file_id,
            output=output_name,
            quiet=False
        )
        print("Archivo descargado desde Google Drive.")

    df_AHP = pd.read_csv(output_name)

except Exception as e:
    print(f"No se pudo descargar/leer desde Drive ({e})")
    df_AHP = pd.read_csv(output_name)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

display(df_AHP.style.hide(axis="index"))

servicios_con_parametros (2).csv ya existe. Se omite la descarga.


Unique_Service,ESI,Dolor,Prob Traslado Hospital,Diagnostico o Situacion
BACK PAINS/INJURY,3,4,0.400000,1
DIABETIC EMERGENCY,3,2,0.600000,1
GAS-ODOR/LEAK,3,1,0.100000,0
CARDIAC EMERGENCY,2,5,0.950000,1
DIZZINESS,4,1,0.300000,1
HEAD INJURY,3,4,0.700000,1
NAUSEA/VOMITING,4,2,0.300000,1
RESPIRATORY EMERGENCY,2,4,0.850000,1
SYNCOPAL EPISODE,3,1,0.600000,1
VEHICLE ACCIDENT -,3,3,0.500000,0


### Filtrado del DataFrame de acuerdo a criterios específicos

Se aplicará un filtro de exclusión sobre las columnas del DataFrame para eliminar aquellos registros que cumplan simultáneamente con todas las siguientes condiciones:
- **ESI** igual a 1, 2, 3.
- **Dolor** igual a 3, 4 y 5.
- **Prob Traslado Hospital** mayor a 0.50.
- **Diagnostico o Situacion** igual a 0.

In [16]:
import pandas as pd
import numpy as np

df_AHP_clean = df_AHP.copy()

df_AHP_clean.columns = df_AHP_clean.iloc[0]

df_AHP_clean = df_AHP_clean.iloc[1:].reset_index(drop=True)

df_AHP_clean['ESI'] = pd.to_numeric(df_AHP_clean['ESI'], errors='coerce')
df_AHP_clean['Dolor'] = pd.to_numeric(df_AHP_clean['Dolor'], errors='coerce')
df_AHP_clean['Prob Traslado Hospital'] = pd.to_numeric(
    df_AHP_clean['Prob Traslado Hospital'], errors='coerce'
)
df_AHP_clean['Diagnostico o Situacion'] = pd.to_numeric(
    df_AHP_clean['Diagnostico o Situacion'], errors='coerce'
)

filtro = (
    (df_AHP_clean['ESI'].isin([1, 2, 3])) |
    (df_AHP_clean['Dolor'].isin([3, 4, 5])) |
    (df_AHP_clean['Prob Traslado Hospital'] > 0.50) |
    (df_AHP_clean['Diagnostico o Situacion'] == 0)
)


df_AHP_filtrado = df_AHP_clean[~filtro].reset_index(drop=True)

print("Filas antes:", len(df_AHP_clean))
print("Filas después:", len(df_AHP_filtrado))

display(df_AHP_filtrado)

Filas antes: 95
Filas después: 4


,Unique_Service,ESI,Dolor,Prob Traslado Hospital,Diagnostico o Situacion
0,DIZZINESS,4,1,0.3,1
1,NAUSEA/VOMITING,4,2,0.3,1
2,GENERAL WEAKNESS,4,1,0.3,1
3,HEAT EXHAUSTION,4,2,0.4,1
